In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Clasificación con DistilRoBERTa

El presente notebook realiza la clasificación de sentimiento de reseñas de IMDB usando DistilRoBERTa.
   - Tokeniza los textos usando el tokenizer de DistilRoBERTa.
   - Realiza Fine-Tuning del modelo preentrenado.

## Instalación y Configuración del Entorno

Instalar las librerías necesarias.
* ***transformers***: Para los modelos y tokenizadores.
* ***datasets***: Para cargar y procesar IMDB.
* ***accelerate***: Para optimizar el entrenamiento.
* ***einops***: Librería auxiliar para la manipulación de tensores.


In [ ]:
!pip install transformers==4.46.3

In [ ]:
!pip install -q datasets accelerate bitsandbytes sentencepiece einops evaluate

In [ ]:
# Cargar librerías
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    pipeline,
    AutoModelForCausalLM
)
from datasets import load_dataset
import evaluate

## Parte 1. Fine-Tuning de DistilRoBERTa en IMDB

Lo primero que se debe realizar es cargar el **dataset IMDB** de *Hugging Face*.

El dataset IMDB se compone de reseñas de películas tanto negativas como positivas (0 = Negativa, 1 = Positiva)

In [ ]:
print("Cargando el dataset IMDB")
imdb_dataset = load_dataset("imdb")

# Se verifica la estructura
print(imdb_dataset)

# Se toma una muestra
print("Ejemplo de reseña")
print(imdb_dataset["train"][0]["text"][:300], "...")
print("Etiqueta:", imdb_dataset["train"][0]["label"])


In [ ]:
# Cargar el Tokenizer de DistilRoBERTa.
model_checkpoint = "distilroberta-base"

print(f"Cargando tokenizer: {model_checkpoint}")
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Se define la función de preprocesamiento.
# Se trunca a 512 (máximo que maneja el modelo).
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

print("Tokenizando dataset")
tokenized_datasets = imdb_dataset.map(tokenize_function, batched=True)

In [ ]:
# Se cargar el modelo para clasificación
print(f"Cargando modelo de clasificación: {model_checkpoint}")
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

Se verifica si el dispositivo en donde se va a trabajar tiene el GPU disponible

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
print(f"Modelo cargado en: {device}")

In [ ]:
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

In [ ]:
# Configurar el entrenamiento con TrainingArguments y Trainer.

# Métrica de precisión.
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# Argumentos de entrenamiento.
training_args = TrainingArguments(
    output_dir="./distilroberta-imdb-classifier",  # Directorio de salida.
    #evaluation_strategy="epoch",                  # Evaluar al final de cada época.
    eval_strategy="epoch",                         # Evaluar al final de cada época.
    save_strategy="epoch",                         # Guardar checkpoint por época.
    learning_rate=2e-5,                            # Tasa de aprendizaje estándar.
    per_device_train_batch_size=16,                # Tamaño del lote según GPU T4.
    per_device_eval_batch_size=64,                 # Lote de evaluación.
    num_train_epochs=3,                            # Número de épocas.
    weight_decay=0.01,                             # Regularización.
    load_best_model_at_end=True,                   # Cargar el mejor modelo al final.
    metric_for_best_model="accuracy",              # Usar 'accuracy' para seleccionar.
    # logging_dir="./logs",                         # Directorio de logs.
    logging_steps=500,                              # Loggear cada 500 pasos.
    report_to="none"                                # Desactiva wandb
)

# Inicializamos el Trainer.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
!# Fine-tuning.
# Atención: El entrenamiento actual es de una época para no requerir mucho tiempo de entrenamiento.

print("Iniciando el fine-tuning")
trainer.train()

In [ ]:
# Guardar el modelo final
print("Guardando el modelo")
trainer.save_model("./distilroberta-imdb-saved")
tokenizer.save_pretrained("./distilroberta-imdb-saved")